In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")

Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from text_processing import *
from utils import *

hotpot_file_candidates = [
    REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json",
    REPO_ROOT / "hotpot_dev_distractor_v1.json",
]
file_path = next((str(path) for path in hotpot_file_candidates if path.exists()), str(hotpot_file_candidates[0]))
print(f"Using HotpotQA file: {file_path}")

NUM_SAMPLES = 200  # Set to None to index the full HotpotQA distractor dev set.
documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=NUM_SAMPLES)

print(f"Documents: {len(documents)}")
print(f"Samples: {len(samples)}")

Using HotpotQA file: /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_dev_distractor_v1.json
Loading cached dataset...
Loaded 1991 documents
Loaded 200 samples
Documents: 1991
Samples: 200


In [3]:
preview_index = 0
print("Question:", samples[preview_index]["question"])
print("Answer:", samples[preview_index]["answer"])
print("Gold doc ids:", samples[preview_index]["gold_doc_ids"])

for doc_id in samples[preview_index]["gold_doc_ids"]:
    doc = documents[doc_id]
    print("\nTitle:", doc["title"])
    print(doc["text"][:500])

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Gold doc ids: [1, 4]

Title: Scott Derrickson
Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."

Title: Ed Wood
Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.


In [4]:
import torch
import RAG_graph

GRAPH_BATCH_SIZE = 64
GRAPH_QUEUE_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_TAG = "all" if NUM_SAMPLES is None else str(NUM_SAMPLES)
INDEX_OUTPUT_DIR = REPO_ROOT / "cache" / "hotpotqa_latest_framework_index"
INDEX_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEX_BASENAME = f"litesemrag_hotpotqa_{DATASET_TAG}"
INDEX_PKL_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}.pkl"
INDEX_TENSOR_PATH = Path(str(INDEX_PKL_PATH).replace(".pkl", "_tensors.pt"))
INDEX_DOCS_JSON_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_documents.json"

ENABLE_PHRASE_AUDIT = True
PHRASE_AUDIT_CACHE_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_phrase_audit.jsonl"
CLEAR_PHRASE_AUDIT_CACHE_ON_REBUILD = True

GRAPH_CONFIG = dict(
    df_ratio=0.9,
    buffer_size=100,
    chunk_size=256,
    device=DEVICE,
    consensus_ratio_threshold=0.8,
    use_llm_candidate_filter=True,
    llm_candidate_filter_use_api=True,
    phrase_audit_enabled=ENABLE_PHRASE_AUDIT,
    phrase_audit_cache_path=str(PHRASE_AUDIT_CACHE_PATH),
)

SAVE_SPLIT_INDEX = True
LOAD_SAVED_INDEX_IF_EXISTS = False
FORCE_REBUILD_INDEX = True
VERIFY_LOAD_AFTER_SAVE = False

print(f"Device: {DEVICE}")
print(f"Index pickle path: {INDEX_PKL_PATH}")
print(f"Index tensor path: {INDEX_TENSOR_PATH}")
print(f"Index document JSON path: {INDEX_DOCS_JSON_PATH}")
print(f"Phrase audit enabled: {ENABLE_PHRASE_AUDIT}")
print(f"Phrase audit cache path: {PHRASE_AUDIT_CACHE_PATH}")
print(f"Load saved index if available: {LOAD_SAVED_INDEX_IF_EXISTS}")
print(f"Force rebuild index: {FORCE_REBUILD_INDEX}")


Device: cuda
Index pickle path: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_200.pkl
Index tensor path: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_200_tensors.pt
Index document JSON path: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_200_documents.json
Phrase audit enabled: True
Phrase audit cache path: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_200_phrase_audit.jsonl
Load saved index if available: False
Force rebuild index: True


In [5]:
def index_cache_exists():
    if SAVE_SPLIT_INDEX:
        return INDEX_PKL_PATH.exists() and INDEX_TENSOR_PATH.exists()
    return INDEX_PKL_PATH.exists()

INDEX_WAS_REBUILT = False
INDEX_CACHE_AVAILABLE = index_cache_exists()

if LOAD_SAVED_INDEX_IF_EXISTS and INDEX_CACHE_AVAILABLE and not FORCE_REBUILD_INDEX:
    print(f"Loading saved index from: {INDEX_PKL_PATH}")
    if SAVE_SPLIT_INDEX:
        graph_database = RAG_graph.LiteSemRAG.load_data_split(str(INDEX_PKL_PATH))
    else:
        graph_database = RAG_graph.LiteSemRAG.load_data(str(INDEX_PKL_PATH))
    graph_database.json_path = str(INDEX_DOCS_JSON_PATH)
    if ENABLE_PHRASE_AUDIT:
        graph_database.enable_phrase_audit(str(PHRASE_AUDIT_CACHE_PATH))
else:
    if FORCE_REBUILD_INDEX and INDEX_CACHE_AVAILABLE:
        print("Force rebuild enabled; existing cache will be overwritten after indexing.")
    elif LOAD_SAVED_INDEX_IF_EXISTS:
        print("No complete saved index cache found; building a new index.")
    else:
        print("Saved-index loading disabled; building a new index.")

    if ENABLE_PHRASE_AUDIT and CLEAR_PHRASE_AUDIT_CACHE_ON_REBUILD and PHRASE_AUDIT_CACHE_PATH.exists():
        PHRASE_AUDIT_CACHE_PATH.unlink()
        print(f"Removed old phrase audit cache: {PHRASE_AUDIT_CACHE_PATH}")

    graph_database = RAG_graph.LiteSemRAG(**GRAPH_CONFIG)
    graph_database.json_path = str(INDEX_DOCS_JSON_PATH)

    graph_database.index_json(
        documents,
        batch_size=GRAPH_BATCH_SIZE,
        queue_size=GRAPH_QUEUE_SIZE,
        sample_count=len(samples),
    )
    graph_database.finalize()
    graph_database.print_memory_size()

    if SAVE_SPLIT_INDEX:
        graph_database.save_data_split(str(INDEX_PKL_PATH))
        print(f"Saved split index to: {INDEX_PKL_PATH}")
        print(f"Saved tensor file to: {INDEX_TENSOR_PATH}")
    else:
        graph_database.save_data(str(INDEX_PKL_PATH))
        print(f"Saved index to: {INDEX_PKL_PATH}")

    INDEX_WAS_REBUILT = True

print(f"Saved indexed document list path: {INDEX_DOCS_JSON_PATH}")
if ENABLE_PHRASE_AUDIT:
    print(f"Phrase audit cache path: {PHRASE_AUDIT_CACHE_PATH}")
print("Graph stats:")
print(f"  loaded from cache: {not INDEX_WAS_REBUILT}")
print(f"  docs: {len(graph_database.doc_nodes)}")
print(f"  chunks: {len(graph_database.chunk_nodes)}")
print(f"  tokens: {len(graph_database.token_nodes)}")
print(f"  phrase tokens: {len(graph_database.phrase_token_nodes)}")
print(f"  sem nodes: {len(graph_database.sem_nodes)}")


Saved-index loading disabled; building a new index.
Removed old phrase audit cache: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_200_phrase_audit.jsonl
Loading text encoder models in device: GPU


/home/xiaoyue/anaconda3/envs/llm_graph/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


CPU preprocessed: 1991/1991 | GPU encoded: 1991/1991 | CPU processed: 1991/1991
semantic_node_build: 66/66 | remaining: 00
[877.3764s] Index 1991 documents. Index pipeline time: 877.3764s
single-cluster semantic build: built 54 sem nodes
[877.3807s] Finalize started.
[877.3818s] Computed average chunk length.
[877.4556s] Removed 613 empty placeholder token nodes.
finalize_token_nodes: 28199/28199 | remaining: 01725
[886.8494s] Finished token node finalization.
merge_duplicate_description_sem_nodes: 10/10
[886.8611s] Finished merging sem nodes by description.
[887.1329s] Finished building modifier postings.
[887.2030s] Finished assigning token and sem IDF.
[887.3359s] Finished computing sem BM25.
[887.5034s] Finished building query database.
[887.5313s] Finished building phrase query index.
[887.6712s] Finished building chunk-to-sem edges.
[887.6735s] Saved sem description logs to /home/xiaoyue/LiteSemRAG/logs/sem_description_20260527_000613.log.
[887.6736s] Finalizing completed.
Index 

In [6]:
import json
import pandas as pd

PHRASE_AUDIT_MAX_ROWS = 10000
PHRASE_AUDIT_FILTER_PROTECTED = None  # Use True, False, or None.
PHRASE_AUDIT_FILTER_REASON = None     # Example: "TITLE_LIKE_NOUN_CHUNK".
PHRASE_AUDIT_FILTER_TEXT = None       # Case-insensitive substring over phrase/raw_text.

PHRASE_AUDIT_SUMMARY_CSV_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_phrase_audit_summary.csv"
PHRASE_AUDIT_ROWS_CSV_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_phrase_audit_rows.csv"
PHRASE_AUDIT_REPORT_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_phrase_audit_report.txt"

phrase_audit_files = sorted(INDEX_OUTPUT_DIR.glob("*phrase_audit*.jsonl"))


phrase_audit_warnings = []


def load_phrase_audit(path=PHRASE_AUDIT_CACHE_PATH):
    path = Path(path)
    phrase_audit_warnings.clear()
    if not path.exists():
        phrase_audit_warnings.append(f"Phrase audit cache not found: {path}")
        return pd.DataFrame()

    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                phrase_audit_warnings.append(f"Skipping invalid JSONL line {line_number}: {exc}")
    return pd.DataFrame(records)


phrase_audit_df = load_phrase_audit()

summary_cols = ["source", "protected", "protected_reason", "entity_label"]
if phrase_audit_df.empty:
    summary_df = pd.DataFrame(columns=summary_cols + ["count"])
    browse_df = phrase_audit_df.copy()
else:
    summary_df = (
        phrase_audit_df
        .groupby(summary_cols, dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    browse_df = phrase_audit_df.copy()
    if PHRASE_AUDIT_FILTER_PROTECTED is not None:
        browse_df = browse_df[browse_df["protected"] == PHRASE_AUDIT_FILTER_PROTECTED]
    if PHRASE_AUDIT_FILTER_REASON:
        browse_df = browse_df[browse_df["protected_reason"] == PHRASE_AUDIT_FILTER_REASON]
    if PHRASE_AUDIT_FILTER_TEXT:
        pattern = str(PHRASE_AUDIT_FILTER_TEXT)
        text_mask = (
            browse_df["phrase"].fillna("").str.contains(pattern, case=False, regex=False)
            | browse_df["raw_text"].fillna("").str.contains(pattern, case=False, regex=False)
        )
        browse_df = browse_df[text_mask]

display_cols = [
    "doc_name",
    "chunk_index",
    "chunk_node_id",
    "phrase",
    "raw_text",
    "source",
    "protected",
    "protected_reason",
    "entity_label",
    "start_char",
    "end_char",
]
available_display_cols = [col for col in display_cols if col in browse_df.columns]
rows_df = browse_df[available_display_cols].head(PHRASE_AUDIT_MAX_ROWS).copy()

summary_df.to_csv(PHRASE_AUDIT_SUMMARY_CSV_PATH, index=False)
rows_df.to_csv(PHRASE_AUDIT_ROWS_CSV_PATH, index=False)

report_lines = [
    f"Loaded phrase audit rows: {len(phrase_audit_df):,}",
    f"Saved summary CSV: {PHRASE_AUDIT_SUMMARY_CSV_PATH}",
    f"Saved rows CSV: {PHRASE_AUDIT_ROWS_CSV_PATH}",
    "Available phrase audit caches:",
]
if phrase_audit_files:
    report_lines.extend(f"  {path} ({path.stat().st_size:,} bytes)" for path in phrase_audit_files)
else:
    report_lines.append("  (none)")
if phrase_audit_warnings:
    report_lines.append("Warnings:")
    report_lines.extend(f"  {warning}" for warning in phrase_audit_warnings)
PHRASE_AUDIT_REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
phrase_audit_output_paths = {
    "summary_csv": PHRASE_AUDIT_SUMMARY_CSV_PATH,
    "rows_csv": PHRASE_AUDIT_ROWS_CSV_PATH,
    "report_txt": PHRASE_AUDIT_REPORT_PATH,
}


In [7]:
from IPython.display import display

MULTI_SEM_MIN_SEM_COUNT = 2
MULTI_SEM_MAX_TOKEN_NODES = 50
MULTI_SEM_MAX_SEMS_PER_TOKEN = 10
MULTI_SEM_MAX_SENTENCES_PER_SEM = 3
MULTI_SEM_MAX_EXAMPLES_PER_TOKEN = 10
MULTI_SEM_TOKEN_CONTAINS = None

multi_sem_token_nodes = [
    token_node
    for token_node in graph_database.token_nodes
    if len(token_node.sem_node_list) >= MULTI_SEM_MIN_SEM_COUNT
]

print(
    f"Token nodes with >= {MULTI_SEM_MIN_SEM_COUNT} sem nodes: "
    f"{len(multi_sem_token_nodes)} / {len(graph_database.token_nodes)}"
)

display(
    graph_database.show_multi_sem_token_nodes(
        min_sem_count=MULTI_SEM_MIN_SEM_COUNT,
        max_sentences_per_sem=MULTI_SEM_MAX_SENTENCES_PER_SEM,
        as_html=True,
        token_contains=MULTI_SEM_TOKEN_CONTAINS,
        sort_by="sem_count",
        max_token_nodes=MULTI_SEM_MAX_TOKEN_NODES,
        max_sems_per_token=MULTI_SEM_MAX_SEMS_PER_TOKEN,
        max_examples_per_token=MULTI_SEM_MAX_EXAMPLES_PER_TOKEN,
        open_details=False,
    )
)


Token nodes with >= 2 sem nodes: 10 / 28199


In [8]:
if VERIFY_LOAD_AFTER_SAVE:
    if INDEX_WAS_REBUILT:
        if SAVE_SPLIT_INDEX:
            loaded_graph = RAG_graph.LiteSemRAG.load_data_split(str(INDEX_PKL_PATH))
        else:
            loaded_graph = RAG_graph.LiteSemRAG.load_data(str(INDEX_PKL_PATH))
        print("Reloaded graph from saved index for verification.")
    else:
        loaded_graph = graph_database
        print("Using graph already loaded from saved index; reload verification skipped.")

    print("Loaded graph stats:")
    print(f"  docs: {len(loaded_graph.doc_nodes)}")
    print(f"  chunks: {len(loaded_graph.chunk_nodes)}")
    print(f"  tokens: {len(loaded_graph.token_nodes)}")
    print(f"  phrase tokens: {len(loaded_graph.phrase_token_nodes)}")
    print(f"  sem nodes: {len(loaded_graph.sem_nodes)}")
    print(f"  query database shape: {None if loaded_graph.query_database is None else tuple(loaded_graph.query_database.shape)}")
